# Lección 4 — Clasificación supervisada end-to-end

**Tema:** el pipeline completo de un modelo supervisado, recorrido de punta a punta. **Train/test split** como primera disciplina, **overfitting** detectado visualmente con `LogisticRegression` + `PolynomialFeatures` sobre un dataset 2D, **normalización** con `StandardScaler` cuando las features están en escalas muy distintas, **pipeline end-to-end** con regresión logística sobre un dataset binario, y simulación de **data drift** para entender cómo se degrada un modelo cuando el mundo cambia.

**Objetivos de esta lección:**
- Aplicar `train_test_split` 80/20 como primer paso del pipeline supervisado, **antes** de cualquier preprocesamiento.
- Reconocer la firma visual del **overfitting** comparando accuracy en train vs test con `LogisticRegression` + `PolynomialFeatures` de complejidad creciente, observando cómo se deforma la frontera de decisión.
- Justificar por qué se normalizan las features con `StandardScaler` antes de entrenar un modelo lineal cuando las columnas están en escalas muy distintas, y medir el impacto sobre la accuracy.
- Recorrer un pipeline supervisado completo: **carga → split → normalización → entrenamiento → evaluación** usando regresión logística.
- Explicar **data drift** como fenómeno de degradación post-despliegue cuando los datos del mundo cambian.

**Side-note:** este notebook usa `numpy`, `pandas`, `matplotlib`, `seaborn` y `scikit-learn`. Todas vienen preinstaladas en Google Colab — no hace falta `pip install`.


## Índice

1. **Train/test split 80/20** — la primera disciplina del pipeline supervisado.
2. **Overfitting visual con LogisticRegression + PolynomialFeatures** — la firma de un modelo que memoriza.
3. **Normalización con `StandardScaler`** — el impacto sobre la accuracy cuando las features están en escalas muy distintas.
4. **Pipeline end-to-end con regresión logística** — los 5 pasos centrales conectados sobre un dataset real.
5. **Data drift simulado** — qué le pasa al modelo cuando el mundo cambia post-despliegue.


In [ ]:
# Celda de setup: ejecutala una sola vez antes de avanzar.
# Todas las librerías vienen preinstaladas en Google Colab — no hace falta pip install.

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_iris, make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score, f1_score, confusion_matrix

# Reproducibilidad: fijamos seed global de numpy y vamos a usar random_state=42
# en todos los métodos estocásticos de sklearn a lo largo del notebook.
np.random.seed(42)

# Estilo de las visualizaciones.
sns.set_theme(style="whitegrid")

print("Setup completo. Todo listo para correr el notebook.")


## 1. Train/test split 80/20

**Idea central:** antes de entrenar cualquier modelo, partimos los datos en dos:
- **Train (80%)** — lo que el modelo ve durante el entrenamiento.
- **Test (20%)** — lo que reservamos como "examen sorpresa" para medir si el modelo generalizó o memorizó.

**Analogía:** es como estudiar para un parcial. Estudiás con un set de ejercicios (train) y te examinás con problemas que **nunca viste antes** (test). Si te dieran el examen final con los mismos ejercicios que estudiaste, no estarían midiendo qué aprendiste — estarían midiendo qué memorizaste.

**Disciplina del pipeline:** el split va **antes** de cualquier transformación (normalización, imputación, encoding). Si normalizamos antes de splitear, le filtramos al modelo información del test set sin querer. Esa regla la vamos a repetir en cada bloque del notebook.


In [ ]:
# Cargamos el dataset Iris: 150 flores, 4 medidas de pétalo y sépalo, 3 especies.
# Es un dataset chico y limpio, perfecto para mostrar la mecánica del split.

iris = load_iris(as_frame=True)
X_iris = iris.data    # matriz de features: 1 fila por flor, 1 columna por medida
y_iris = iris.target  # vector target: la especie de cada flor (0/1/2)

# Vista 1 — el dataset completo (features + target en una sola tabla).
df_iris = X_iris.copy()
df_iris["especie"] = y_iris
print(f"Dataset completo: {df_iris.shape[0]} filas × {df_iris.shape[1]} columnas")
print(f"  → X (features): {X_iris.shape}   |   y (target): {y_iris.shape}")
print(f"  → Clases (especies): {list(iris.target_names)}")
df_iris.head()


**Cómo se separan los datos en X y y:**

- **`X` — la matriz de features.** Cada **fila** es un ejemplo (una flor), cada **columna** es una variable medida (largo de pétalo, ancho de sépalo, etc.). En este dataset: `(150, 4)` → 150 flores × 4 medidas.
- **`y` — el vector target.** Un valor por cada fila de `X`, alineado por posición. La fila `i` de `X` describe a la flor cuya especie está en `y[i]`. Acá: `(150,)` → 150 etiquetas, una por flor.

Mirá las dos vistas por separado para que quede claro:


In [ ]:
# Vista 2 — solo la matriz X (features).
print(f"Dimensiones de X: {X_iris.shape}   ({X_iris.shape[0]} ejemplos, {X_iris.shape[1]} features)")
print(f"Nombres de las columnas (features): {list(X_iris.columns)}")
print()
print("Primeras 5 filas de X:")
X_iris.head()


Cada fila es una flor; las 4 columnas son las medidas en cm. Esta es la "entrada" que va a recibir el modelo.


In [ ]:
# Vista 3 — solo el vector y (target).
print(f"Dimensiones de y: {y_iris.shape}   ({y_iris.shape[0]} etiquetas, una por fila de X)")
print(f"Valores únicos en y: {sorted(y_iris.unique())}   (corresponden a las 3 especies)")
print()
print("Primeros 10 valores de y:")
print(y_iris.head(10).to_list())


Cada número es la especie de la flor correspondiente: `0 = setosa`, `1 = versicolor`, `2 = virginica`. Esto es lo que el modelo va a aprender a predecir a partir de las medidas en X.


In [ ]:
# Aplicamos train_test_split: 80% para entrenamiento, 20% para testeo.
# random_state=42 fija la aleatoriedad — así siempre vemos los mismos números.

X_train_iris, X_test_iris, y_train_iris, y_test_iris = train_test_split(
    X_iris, y_iris, test_size=0.2, random_state=42
)

print(f"X_train: {X_train_iris.shape}   (80% de las filas)")
print(f"X_test:  {X_test_iris.shape}    (20% reservadas para evaluar)")
print(f"y_train: {y_train_iris.shape}")
print(f"y_test:  {y_test_iris.shape}")


**Lo que pasó:** la función nos devolvió 4 arrays. `X_train` y `y_train` son las features y etiquetas que el modelo va a ver durante el entrenamiento. `X_test` e `y_test` son los datos que reservamos para evaluarlo después — el modelo NO los va a ver durante el `fit`.


### Ejercicio — la trampa de evaluar con el train set

¿Qué pasa si entrenamos con el 100% de los datos y los evaluamos con esos MISMOS datos?

a) El modelo va a parecer mejor de lo que realmente es.

b) El modelo va a fallar al ejecutarse.

c) No pasa nada.

d) El modelo va a tardar más en entrenar.

<details>
<summary>Resultado esperado</summary>

**Respuesta correcta: (a).** El modelo "memoriza" en lugar de aprender. Su accuracy contra los mismos datos que vio durante el entrenamiento puede ser muy alta — pero ese número no nos dice **nada** sobre cómo se va a comportar en datos nuevos. Por eso siempre reservamos un test set: necesitamos un examen genuinamente "no visto" para medir generalización honesta.

</details>


## 2. El problema de Overfitting

**Idea central:** un modelo puede sacarle "10" al train y "2" al test. Eso se llama **overfitting** (sobreajuste): el modelo memorizó los datos de entrenamiento en vez de aprender la regla que los gobierna.

Vamos a verlo con los ojos. Construimos un dataset 2D de clasificación binaria (dos features, dos clases) y ajustamos **tres `LogisticRegression` de complejidad creciente** sobre el mismo train set. Para que la frontera de decisión se vuelva más flexible, expandimos las features con `PolynomialFeatures` de grado 1, 3 y 15. La idea es ver cómo cambia la **frontera** que aprende el modelo a medida que le permitimos que sea más enrulada.

![alt text](overfitting.png)

In [ ]:
# Generamos un dataset 2D no-linealmente separable: dos lunas entrelazadas.
# La forma de "luna" hace que ninguna recta lo separe perfectamente —
# justo lo que necesitamos para ver cómo se comportan modelos de distinta complejidad.

from sklearn.datasets import make_moons

X_of, y_of = make_moons(n_samples=200, noise=0.30, random_state=42)

# Split 80/20 antes de cualquier transformación (la disciplina del bloque 1).
X_train_of, X_test_of, y_train_of, y_test_of = train_test_split(
    X_of, y_of, test_size=0.2, random_state=42
)

# Visualizamos el dataset crudo antes de modelar.
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_train_of[:, 0], X_train_of[:, 1], c=y_train_of, cmap="coolwarm",
           s=55, alpha=0.85, label="train", edgecolor="white", linewidth=0.5)
ax.scatter(X_test_of[:, 0], X_test_of[:, 1], c=y_test_of, cmap="coolwarm",
           s=90, marker="s", alpha=0.95, label="test",
           edgecolor="black", linewidth=0.8)
ax.set_xlabel("feature 1")
ax.set_ylabel("feature 2")
ax.set_title("Dataset 2D: dos clases con forma de lunas entrelazadas")
ax.legend()
plt.tight_layout()
plt.show()


**¿Qué se ve?** Dos nubes de puntos en forma de luna, una azul y otra roja, parcialmente entrelazadas. **No hay ninguna recta** que las separe perfectamente — para acertar bien va a hacer falta una frontera curva. Ahora veamos qué pasa cuando entrenamos `LogisticRegression` con tres niveles de flexibilidad distintos.


In [ ]:
# Ajustamos LogisticRegression con PolynomialFeatures de grado 1, 3 y 15.

grados = [1, 3, 15]
configs = {
    1:  {"C": 1.0,   "label": "grado 1 (frontera lineal)"},
    3:  {"C": 1e3,   "label": "grado 3 (frontera curva)"},
    15: {"C": 1e10,  "label": "grado 15 (frontera retorcida)"},
}

modelos_of = {}
acc_resultados = []

for g in grados:
    poly_g = PolynomialFeatures(degree=g, include_bias=False)
    X_train_g = poly_g.fit_transform(X_train_of)
    X_test_g = poly_g.transform(X_test_of)

    modelo_g = LogisticRegression(C=configs[g]["C"], max_iter=5000, random_state=42)
    modelo_g.fit(X_train_g, y_train_of)

    acc_train = accuracy_score(y_train_of, modelo_g.predict(X_train_g))
    acc_test = accuracy_score(y_test_of, modelo_g.predict(X_test_g))

    modelos_of[g] = (poly_g, modelo_g)
    acc_resultados.append({"grado": g, "acc_train": acc_train, "acc_test": acc_test})

# Graficamos las 3 fronteras de decisión lado a lado.
fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), sharey=True)

x_min, x_max = X_of[:, 0].min() - 0.5, X_of[:, 0].max() + 0.5
y_min, y_max = X_of[:, 1].min() - 0.5, X_of[:, 1].max() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300),
                     np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

for ax, g in zip(axes, grados):
    poly_g, modelo_g = modelos_of[g]
    Z = modelo_g.predict(poly_g.transform(grid)).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.30, cmap="coolwarm", levels=[-0.5, 0.5, 1.5])
    ax.scatter(X_train_of[:, 0], X_train_of[:, 1], c=y_train_of, cmap="coolwarm",
               s=45, alpha=0.85, edgecolor="white", linewidth=0.5, label="train")
    ax.scatter(X_test_of[:, 0], X_test_of[:, 1], c=y_test_of, cmap="coolwarm",
               s=80, marker="s", alpha=0.95, edgecolor="black", linewidth=0.8, label="test")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)
    ax.set_xlabel("feature 1")
    ax.set_title(configs[g]["label"])
    ax.legend(loc="upper right", fontsize=9)

axes[0].set_ylabel("feature 2")
plt.suptitle("Tres LogisticRegression de complejidad creciente sobre el mismo train set",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


**¿Qué mirar?**

- **Grado 1** (frontera lineal): demasiado simple. Como las lunas no se separan con una recta, el modelo deja muchos puntos del lado equivocado. Está **subajustado** (underfitting).
- **Grado 3**: la frontera se curva con suavidad y sigue la forma de las lunas. Pasa cerca de la verdadera regla sin obsesionarse con cada punto.
- **Grado 15**: la frontera se vuelve retorcida y aparecen "islas" de un color rodeadas por otro. El modelo se contorsiona para capturar **cada punto individual de train**, incluyendo los ruidosos. Esa es la firma visual del **overfitting**.

Ahora veamos las accuracy en train y test para confirmar lo que vemos a ojo.


In [ ]:
# Tabla resumen: accuracy en train y test para cada grado.

tabla_acc = pd.DataFrame(acc_resultados)
tabla_acc["diferencia"] = tabla_acc["acc_train"] - tabla_acc["acc_test"]
tabla_acc.style.format({"acc_train": "{:.3f}", "acc_test": "{:.3f}", "diferencia": "{:+.3f}"})


**Lectura fila por fila:**

- **Grado 1:** accuracy parecida en train y test, pero ambas relativamente bajas. Modelo subajustado — ni siquiera aprende lo que hay.
- **Grado 3:** accuracy alta en train, **similar** en test. **Sweet spot** — el modelo generalizó.
- **Grado 15:** accuracy casi perfecta en train, pero en test cae bastante. La diferencia train-vs-test crece. **Overfitting** — el modelo memorizó cada punto del train (incluyendo el ruido), así que falla cuando ve datos nuevos.


**Ejemplo de overfitting en regresión**

![alt text](overfitting-regression.png)

### Ejercicio — elegir el modelo para producción

Tenés que elegir uno de estos dos modelos para poner en producción.

- **Modelo A:** accuracy train = 0.91, accuracy test = 0.89.
- **Modelo B:** accuracy train = 1.00, accuracy test = 0.72.

¿Cuál usás?

a) Modelo A.

b) Modelo B (tiene accuracy de train más alta).

c) Cualquiera de los dos, da igual.

<details>
<summary>Resultado esperado</summary>

**Respuesta correcta: (a) — Modelo A.**

El modelo A **generaliza**: su performance en train (0.91) y test (0.89) es prácticamente la misma. Eso significa que aprendió la regla, no los datos puntuales.

El modelo B **memorizó**: tiene accuracy perfecta en train, pero en test cae a 0.72. Cuando lo pongas en producción y le lleguen datos nuevos, va a fallar como el grado 15 del ejemplo anterior.

**Take-away — la firma del overfitting:** accuracy alta en train **+** accuracy más baja en test. Si solo mirás performance en train, te engañás. Por eso siempre, siempre, siempre evaluamos en un test set que el modelo no vio.

</details>


## 3. Normalización

**Caso real:** querés predecir si un cliente va a aceptar una oferta comercial. Tenés 5 datos del cliente: `edad` (entre 18 y 80), `ingreso_mensual` (entre 50 mil y 500 mil pesos), `antiguedad_meses` (entre 0 y 120), `productos_contratados` (entre 1 y 10) y `score_navegacion` (entre 0 y 100). Las columnas están en escalas muy distintas — `ingreso_mensual` vale cientos de miles mientras que `productos_contratados` vale unidades.

**Problema:** muchos modelos se "obsesionan" con la columna de números más grandes y subutilizan a las demás, aunque las otras sean igual o más informativas. El modelo termina mirando casi sólo el ingreso e ignorando el resto.

**Analogía:** imaginate un jurado en el que un miembro grita y los demás susurran. El que grita se hace escuchar aunque diga lo mismo (o menos) que los demás. **Normalizar** las features es el equivalente a pedirle a todos que hablen al mismo volumen — recién ahí podemos comparar lo que cada uno aporta.

**Qué hace `StandardScaler`:** lleva cada feature a media 0 y desvío 1. Después de aplicarlo, todas las columnas quedan en una escala comparable.


In [ ]:
# Generamos un dataset sintético pero con nombres y escalas realistas:
# predecir si un cliente va a aceptar una oferta comercial.

X_norm_raw, y_norm = make_classification(
    n_samples=400, n_features=5, n_informative=4, n_redundant=1,
    random_state=42, class_sep=1.0,
)

# Ajustamos cada feature a una escala realista del mundo real.
X_norm = X_norm_raw.copy()
X_norm[:, 0] = X_norm[:, 0] * 15 + 45              # edad (años)
X_norm[:, 1] = X_norm[:, 1] * 80000 + 200000       # ingreso_mensual (pesos)
X_norm[:, 2] = X_norm[:, 2] * 30 + 50              # antiguedad_meses
X_norm[:, 3] = (X_norm[:, 3] * 2 + 4).round()      # productos_contratados
X_norm[:, 4] = X_norm[:, 4] * 20 + 50              # score_navegacion (0-100)

cols_norm = ["edad", "ingreso_mensual", "antiguedad_meses",
             "productos_contratados", "score_navegacion"]

# Split antes de cualquier transformación (la disciplina del bloque 1).
X_train_norm, X_test_norm, y_train_norm, y_test_norm = train_test_split(
    X_norm, y_norm, test_size=0.2, random_state=42
)

# Mostramos las primeras filas para que se vean las escalas tan distintas.
print("Primeras 3 filas de X_train (mirá las escalas tan distintas entre columnas):")
pd.DataFrame(X_train_norm, columns=cols_norm).head(3).round(2)


**Lo que se ve:** `ingreso_mensual` está en escala cientos de miles, mientras que `edad` y `antiguedad_meses` están en decenas y `productos_contratados` en unidades. Esa diferencia de escalas es muy común en datasets reales — basta con tener una columna en pesos al lado de otra en cantidad de productos.

Veamos qué accuracy logra un modelo de clasificación entrenado **sin** normalizar.


In [ ]:
# Entrenamos LogisticRegression directamente, sin normalizar las features.

modelo_sin = LogisticRegression(solver="saga", max_iter=10000, random_state=42)
modelo_sin.fit(X_train_norm, y_train_norm)
acc_sin = modelo_sin.score(X_test_norm, y_test_norm)

print(f"Accuracy en test (sin normalizar): {acc_sin:.4f}")


**Lo que pasó:** la accuracy es muy baja, prácticamente como tirar una moneda. El modelo quedó "dominado" por `ingreso_mensual` y no logró aprovechar las otras 4 features porque sus números eran chiquitos al lado del ingreso.


### Ejercicio — anticipá el resultado

Si normalizamos las 5 features con `StandardScaler` y volvemos a entrenar la misma `LogisticRegression`, ¿qué esperás ver al comparar la accuracy con la del modelo sin normalizar?

a) Igual o un poquito mejor.

b) Bastante mejor — la accuracy va a subir varios puntos.

c) Mucho peor.

d) Va a tirar error.

<details>
<summary>Resultado esperado</summary>

**Respuesta correcta: (b).**

Al llevar las 5 features a la misma escala, el modelo deja de estar dominado por `ingreso_mensual` y puede aprovechar la información de las otras 4 columnas. La accuracy en test sube de manera **clara**, varios puntos.

</details>


In [ ]:
# Aplicamos StandardScaler: fit en TRAIN, transform en train Y test.
# Nunca al revés — fit-ear sobre test sería filtrarle al scaler info que no debería ver.

scaler_norm = StandardScaler()
X_train_norm_s = scaler_norm.fit_transform(X_train_norm)
X_test_norm_s = scaler_norm.transform(X_test_norm)

# Entrenamos el MISMO modelo sobre los datos normalizados.
modelo_con = LogisticRegression(solver="saga", max_iter=2000, random_state=42)
modelo_con.fit(X_train_norm_s, y_train_norm)
acc_con = modelo_con.score(X_test_norm_s, y_test_norm)

print(f"Accuracy SIN normalizar: {acc_sin:.4f}")
print(f"Accuracy CON normalizar: {acc_con:.4f}")
print(f"Diferencia:              {acc_con - acc_sin:+.4f}")


**Lo que cambió:** la accuracy sube **muchísimo** — del nivel de tirar una moneda a un modelo razonablemente útil. Mismo algoritmo, mismo dataset, mismo split. La única diferencia es que ahora todas las features están en la misma escala y el modelo puede aprovecharlas a todas.

Visualicemos esa diferencia para que quede grabada.


In [ ]:
# Comparamos las dos accuracies en un bar chart.
fig, ax = plt.subplots(figsize=(7, 5))

ax.bar(["sin normalizar", "normalizado"],
       [acc_sin, acc_con],
       color=["C3", "C2"],
       alpha=0.85, edgecolor="black", linewidth=0.8)

for i, v in enumerate([acc_sin, acc_con]):
    ax.text(i, v + 0.02, f"{v:.3f}", ha="center", fontsize=12, fontweight="bold")

ax.set_ylabel("accuracy en test")
ax.set_ylim(0, 1.0)
ax.set_title("Impacto de normalizar las features")
plt.tight_layout()
plt.show()


**Take-away:** cuando las features tienen escalas muy distintas, normalizarlas antes de entrenar puede mejorar dramáticamente la accuracy. La regla práctica: **siempre normalizar** las features numéricas antes de entrenar un modelo. Es barato, es seguro, y a veces — como en este ejemplo — rescata decenas de puntos de performance.

Más adelante vas a ver otro motivo (totalmente distinto) para normalizar: los algoritmos basados en distancia entre puntos.


## 4. Pipeline end-to-end con regresión logística

**Idea central:** hasta acá vimos las etapas por separado. Ahora las juntamos en un solo bloque.

El ciclo de un proyecto de ML tiene **7 pasos**:

1. **Recogida de datos**.
2. **Preprocesamiento de datos**.
3. **Elegir el modelo adecuado**.
4. **Entrenamiento del modelo**.
5. **Evaluar el modelo**.
6. **Ajuste y optimización de hiperparámetros**.
7. **Predicciones e implementación**.

En este pipeline mínimo viable vamos a ver **explícitamente en código** los pasos **1, 2, 4 y 5** (más un chequeo de overfitting dentro del paso 5). El **paso 3** queda implícito en una sola línea — elegimos el modelo cuando lo instanciamos. Los pasos **6 y 7** los dejamos comentados como "fuera de scope" de este notebook: los vas a ver más adelante (búsqueda en grilla, validación cruzada, MLOps).

Acá tenés el pipeline completo en una sola celda — el "mapa" que vas a leer y escribir todos los días cuando trabajes con modelos supervisados.


In [ ]:
# Pipeline supervisado mínimo viable: los 7 pasos teóricos en una sola celda.
# Los pasos 1, 2, 4 y 5 se ejecutan en código. Los pasos 3, 6 y 7 quedan
# marcados con un comentario que explica por qué no se ven (o se ven sólo en parte) acá.

# PASO 1 — Recogida de datos.
# (Acá generamos un dataset sintético; en producción vendría de un CSV, de una API o de scraping.)
X_pipe, y_pipe = make_classification(
    n_samples=800, n_features=5, n_informative=4, n_redundant=1, random_state=42
)

# PASO 2 — Preprocesamiento de datos.
# este paso incluye limpieza, imputación de valores faltantes y tratamiento de outliers; acá el
# dataset sintético ya viene limpio, así que el preprocesamiento se reduce a split + normalización.
# (El split SIEMPRE va antes de normalizar — si no, le filtramos info del test al scaler.)
X_train_pipe, X_test_pipe, y_train_pipe, y_test_pipe = train_test_split(
    X_pipe, y_pipe, test_size=0.2, random_state=42
)
scaler_pipe = StandardScaler()
X_train_pipe_scaled = scaler_pipe.fit_transform(X_train_pipe)
X_test_pipe_scaled = scaler_pipe.transform(X_test_pipe)

# PASO 3 — Elegir el modelo adecuado.
# Usamos LogisticRegression: clasificación binaria, lineal, rápida e interpretable.
# Es un baseline razonable antes de probar algo más sofisticado.
model_pipe = LogisticRegression(random_state=42, max_iter=1000)

# PASO 4 — Entrenamiento del modelo.
model_pipe.fit(X_train_pipe_scaled, y_train_pipe)

# PASO 5 — Evaluar el modelo (con métricas sobre el test normalizado).
y_pred_pipe = model_pipe.predict(X_test_pipe_scaled)
acc_pipe = accuracy_score(y_test_pipe, y_pred_pipe)
f1_pipe = f1_score(y_test_pipe, y_pred_pipe)
cm_pipe = confusion_matrix(y_test_pipe, y_pred_pipe)

# Chequeo final de overfitting (parte del paso 5): accuracy en train vs test.
acc_train_pipe = model_pipe.score(X_train_pipe_scaled, y_train_pipe)

# PASO 6 — Ajuste y optimización de hiperparámetros: lo omitimos en este pipeline mínimo;
# lo vas a ver más adelante con búsqueda en grilla (GridSearchCV) y validación cruzada.

# PASO 7 — Predicciones e implementación: el `predict` de arriba ya cubre la parte de predicciones;
# el despliegue en producción (MLOps) queda fuera del alcance de este notebook.

print(f"Accuracy test:  {acc_pipe:.4f}")
print(f"F1 test:        {f1_pipe:.4f}")
print(f"Accuracy train: {acc_train_pipe:.4f}   (diferencia con test: {acc_train_pipe - acc_pipe:+.4f})")
print()
print("Confusion matrix:")
print(pd.DataFrame(
    cm_pipe,
    index=["Real: 0", "Real: 1"],
    columns=["Pred: 0", "Pred: 1"],
))


**Eso es todo:** los 7 pasos en una sola celda. Cuatro de ellos (1, 2, 4 y 5) se traducen directamente a código y suman ~15 líneas; el paso 3 se reduce a elegir qué modelo instanciar, y los pasos 6 y 7 quedan como comentarios — no porque no importen, sino porque son materia de lecciones más avanzadas. Este patrón se repite con cualquier modelo supervisado: cuando lo tengas internalizado, vas a poder armar el esqueleto de cualquier proyecto de clasificación supervisada en minutos.


## 5. Data drift simulado

**Caso real:** una empresa entrenó en 2019 un modelo que predice si un cliente va a aceptar una oferta de crédito, basándose en sus patrones de gasto: cuánto gasta en restaurantes, en supermercado, en transporte, en entretenimiento. El modelo funcionó muy bien en producción durante todo un año.

Llegó **marzo de 2020** y la pandemia: la gente cambió abruptamente sus hábitos — desaparecieron los gastos en restaurantes y transporte, explotaron los gastos en delivery y compras online, aparecieron cuotas atrasadas que antes no había. Sin que nadie haya tocado el código ni el modelo, **la accuracy en producción se desplomó**: el modelo predecía bien para un mundo que ya no existía.

**Idea central:** un modelo no se "rompe" técnicamente — el mundo cambia. Los datos nuevos ya no se parecen a los datos con los que el modelo aprendió. Eso se llama **data drift** y es la razón principal por la que los sistemas profesionales de Machine Learning **monitorean accuracy en producción de manera permanente**, no sólo durante el desarrollo.

Vamos a simular ese fenómeno tomando el modelo del bloque anterior — `model_pipe`, ya entrenado — y movemos los datos del test simulando que pasó el tiempo y el mundo cambió. NO vamos a tocar el modelo: sólo movemos los datos.


In [ ]:
# Punto de partida: el modelo del pipeline evaluado en condiciones "normales".
# Esto ya lo calculamos en el bloque anterior, lo recuperamos como baseline.

acc_baseline = accuracy_score(y_test_pipe, model_pipe.predict(X_test_pipe_scaled))
print(f"Accuracy baseline (sin drift): {acc_baseline:.4f}")


Ese 0.85+ es la accuracy "del día 1" — el modelo recién puesto en producción, evaluado contra datos del mismo origen que los de entrenamiento. Ahora simulemos que pasa el tiempo y los datos nuevos empiezan a diferir.


In [ ]:
# Simulamos drift: agregamos un offset a las features de test ANTES de aplicar el scaler.
# El offset representa que las distribuciones de entrada se corrieron en el mundo real.

offset_drift = np.array([2.0, 1.5, 0.0, 0.0, 0.0])
X_test_drifted = scaler_pipe.transform(X_test_pipe + offset_drift)

# Aplicamos el MISMO modelo entrenado, sin reentrenar.
acc_drifted = accuracy_score(y_test_pipe, model_pipe.predict(X_test_drifted))

print(f"Accuracy baseline (sin drift): {acc_baseline:.4f}")
print(f"Accuracy con drift:            {acc_drifted:.4f}")
print(f"Caída:                         {(acc_baseline - acc_drifted)*100:.1f} puntos porcentuales")


**Lo que pasó:** mismo modelo, mismas etiquetas reales, **datos de entrada distintos** — y la accuracy cae varios puntos porcentuales. El modelo no se equivocó "por ser malo": el contexto se movió.


In [ ]:
# Curva de degradación: graficamos accuracy a lo largo de niveles crecientes de drift.
# Multiplicamos el offset por un factor que crece de 0 (sin drift) a 3.0 (drift severo).

niveles = np.linspace(0, 3.0, 10)
accs_drift = []

for k in niveles:
    X_drift_k = scaler_pipe.transform(X_test_pipe + k * offset_drift)
    acc_k = accuracy_score(y_test_pipe, model_pipe.predict(X_drift_k))
    accs_drift.append(acc_k)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(niveles, accs_drift, marker="o", linewidth=2, color="C3")
ax.axhline(acc_baseline, color="C2", linestyle="--", linewidth=1.5,
           label=f"baseline = {acc_baseline:.2f}")
ax.set_xlabel("nivel de drift (multiplicador del offset)")
ax.set_ylabel("accuracy en test")
ax.set_title("Degradación del modelo a medida que el mundo se mueve")
ax.set_ylim(0.4, 1.0)
ax.legend()
plt.tight_layout()
plt.show()


**Lo que se ve:** una curva claramente descendente. A más drift, peor accuracy. **El modelo no cambió** — los pesos siguen siendo los mismos, los hiperparámetros siguen siendo los mismos. Cambió la distribución de los datos de entrada, y la performance se cae sola.


### Ejercicio — diagnosticar drift en producción

Tu modelo de scoring crediticio tenía **92% de accuracy hace 6 meses** cuando lo lanzaste a producción. Ahora, sin que nadie haya tocado el código ni el modelo, está rindiendo al **71%**. ¿Qué hipótesis es la más probable?

a) El modelo se "rompió" técnicamente — algo del software cambió.

b) Los datos del mundo cambiaron — drift.

c) Hace falta más entrenamiento sobre los datos viejos.

<details>
<summary>Resultado esperado</summary>

**Respuesta correcta: (b) — drift.**

Una caída tan grande sin tocar el código apunta a que la distribución de los datos de entrada cambió: pueden haber cambiado los hábitos de consumo de los clientes, la situación macroeconómica, el perfil de los nuevos usuarios, los productos crediticios ofrecidos, etc.

- **(a)** es poco probable: si fuera un bug, lo más común es que el modelo dé errores explícitos, no que baje 21 puntos de accuracy de manera silenciosa.
- **(c)** **no resuelve drift**: entrenar más sobre los datos viejos consolida la regla vieja, que es justamente la que dejó de servir. Lo que hay que hacer es **reentrenar con datos actuales**.

**Take-away:** detectar drift requiere monitoreo continuo en producción. No se ve en el test set inicial — el test set es del mismo momento que el train. Por eso los sistemas profesionales miden la accuracy del modelo en producción de manera permanente, y existen estándares específicos para gestionar el ciclo de vida de modelos de IA.

</details>


## Resumen

- **X e y por separado**: el dataset se descompone en una matriz de features `X` (filas = ejemplos, columnas = variables) y un vector target `y` alineado por posición. Cada modelo (supervisado) parte de ese formato.
- **Train/test split 80/20**: va antes de cualquier transformación. Sin un test set "no visto", no podemos saber si el modelo generaliza o memoriza.
- **Overfitting**: el modelo memoriza datos de entrenamiento y no generaliza bien con datos reales no vistos. Ejemplo en clasificación: accuracy alta en train, pero accuracy baja en test.
- **Normalización**: cuando las features están en escalas muy distintas, algunos modelos quedan dominados por las columnas con números más grandes y subutilizan al resto. Normalizar permite llevarlas a una escala parecida, y puede mejorar la velocidad de conversión, e incluso la accuracy de manera dramática.
- **Pipeline end-to-end**: cargar → split → normalizar → entrenar → evaluar
- **Data drift** — cuando el mundo cambia post-despliegue, el modelo se degrada sin que nada esté roto técnicamente. Detectarlo requiere monitoreo continuo en producción.
